# 006 - Random Forest

## Objetivo

Testar uma Random Forest para prever `AGE_RANGE`, com as mesmas variáveis e a mesma divisão estratificada 80/20 (`random_state=42`) das experiências anteriores. A configuração inicial usa 200 árvores. A seleção de `max_depth` e `min_samples_leaf` é feita por Macro F1 em 3 folds estratificados apenas na parcela de treino; a validação fixa só é avaliada depois da seleção.

Referências na divisão fixa: regressão logística regularizada do 002, Macro F1 **0,4125**; árvore ajustada do 005, **0,4037**. Este notebook regista a avaliação e não cria uma submissão.

## Ambiente, dados e divisão fixa

Usamos a preparação central em `src/preparar_dados.py`, os CSV originais e a mesma divisão (`test_size=0.2`, `random_state=42`). A validação fica reservada durante a pesquisa. O pré-processador é incluído na pipeline para ser ajustado separadamente dentro de cada fold e, posteriormente, no treino completo da divisão antes da avaliação fixa.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import sklearn

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')

sys.path.insert(0, str(repo_root))
from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)

assert len(X_train) == 8769 and len(X_val) == 2193, (len(X_train), len(X_val))
assert X_train.columns.equals(X_val.columns), 'As colunas de treino e validação não estão alinhadas.'
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')

## Pesquisa de hiperparâmetros no treino

Testamos as seis combinações de `max_depth=[10, 20, None]` e `min_samples_leaf=[2, 5]`, mantendo `n_estimators=200` e os restantes parâmetros padrão relevantes da Random Forest. Os folds são estratificados, com `shuffle=True` e `random_state=42`. `n_jobs=1` limita o consumo de memória tanto na pesquisa como no estimador. Cada ajuste inclui um pré-processador novo na pipeline, sem usar a validação fixa.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

rf_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rf_search = GridSearchCV(
    estimator=make_model_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            n_jobs=1,
            random_state=42,
        ),
        normalize_numeric=False,
    ),
    param_grid={
        'model__max_depth': [10, 20, None],
        'model__min_samples_leaf': [2, 5],
    },
    scoring='f1_macro',
    cv=rf_cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    error_score='raise',
    verbose=1,
)
rf_search.fit(X_train, y_train)

rf_results = pd.DataFrame(rf_search.cv_results_)
rf_table = rf_results[[
    'param_model__max_depth',
    'param_model__min_samples_leaf',
    'mean_train_score',
    'mean_test_score',
    'std_test_score',
    'rank_test_score',
    'split0_test_score',
    'split1_test_score',
    'split2_test_score',
]].rename(columns={
    'param_model__max_depth': 'max_depth',
    'param_model__min_samples_leaf': 'min_samples_leaf',
    'mean_train_score': 'treino_macro_f1_medio',
    'mean_test_score': 'cv_macro_f1_medio',
    'std_test_score': 'cv_macro_f1_desvio_padrao',
    'rank_test_score': 'posicao',
    'split0_test_score': 'fold_1',
    'split1_test_score': 'fold_2',
    'split2_test_score': 'fold_3',
}).sort_values(['posicao', 'max_depth', 'min_samples_leaf'])
display(rf_table.reset_index(drop=True))

best_params = {
    'max_depth': rf_search.best_params_['model__max_depth'],
    'min_samples_leaf': rf_search.best_params_['model__min_samples_leaf'],
}
best_cv_macro_f1 = rf_search.best_score_
print(f'Melhores parâmetros por CV: {best_params}')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')
print(f'Macro F1 treino CV média: {rf_search.cv_results_["mean_train_score"][rf_search.best_index_]:.4f}')

## Avaliação na validação fixa

Depois de selecionar os parâmetros pela CV, usamos o estimador reajustado em todo `X_train` para prever `X_val`. Reportamos Macro F1 com as 16 classes incluídas e comparamos diretamente com o 002 (**0,4125**) e o 005 (**0,4037**). A validação não participou da escolha dos hiperparâmetros.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_rf_pipe = rf_search.best_estimator_
rf_train_pred = best_rf_pipe.predict(X_train)
rf_val_pred = best_rf_pipe.predict(X_val)
rf_train_macro_f1 = f1_score(
    y_train, rf_train_pred, labels=list(range(16)), average='macro', zero_division=0
)
rf_val_macro_f1 = f1_score(
    y_val, rf_val_pred, labels=list(range(16)), average='macro', zero_division=0
)
logistic_002_macro_f1 = 0.4125
tree_005_macro_f1 = 0.4037

print(f'Parâmetros selecionados: {best_params}; n_estimators=200; n_jobs=1')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')
print(f'Macro F1 treino: {rf_train_macro_f1:.4f}')
print(f'Macro F1 validação: {rf_val_macro_f1:.4f}')
print(f'Diferença vs. regressão logística 002 (0.4125): {rf_val_macro_f1 - logistic_002_macro_f1:+.4f}')
print(f'Diferença vs. árvore ajustada 005 (0.4037): {rf_val_macro_f1 - tree_005_macro_f1:+.4f}')
print('--- classificação por classe ---')
print(classification_report(
    y_val,
    rf_val_pred,
    labels=list(range(16)),
    zero_division=0,
    target_names=[str(i) for i in range(16)],
))
rf_confusion = confusion_matrix(y_val, rf_val_pred, labels=list(range(16)))
display(pd.DataFrame(rf_confusion, index=range(16), columns=range(16)))

## Registo e decisão

Preencher após executar a pesquisa:

- Melhor `max_depth` / `min_samples_leaf`:
- Macro F1 média e desvio padrão nos três folds:
- Macro F1 de treino / validação fixa:
- Diferença para 002 (0,4125):
- Diferença para 005 (0,4037):
- Decisão sobre uma eventual submissão:

Não é gerada submissão neste notebook. A decisão de produzir um CSV será tomada depois de rever estes resultados.